In [ ]:
# just for google colab
from google.colab import drive
drive.mount('/content/drive')

#!git clone https://github.com/TomLi515/Campus_Life_Coach.git

In [ ]:
# Testing clearing caches
import sys
sys.modules.pop('finetune.models', None)
sys.modules.pop('finetune', None)

In [1]:
# Inference initial definitions
# Our sample rate was 50 Hz so 50 samples per second
# I chose 2s window length so we will have 100 samples per guess
# We will do a 1s hop size so we will do 50% overlap
# [a,b,c,d] first sample is ab second is 1 sec ahead and is bc the cd
#model = models.py
#weights = fusion_classifier.pth,phone_classifier.pth,watch_classifier.pth
# Length in seconds
WINDOW_LENGTH = 3
HOP_LENGTH = 1
# What we sampled
SAMPLE_RATE = 50
# Actual converted length
WINDOW_SAMPLES = int(WINDOW_LENGTH * SAMPLE_RATE)
HOP_SAMPLES = int(HOP_LENGTH * SAMPLE_RATE)
# This is from finetuning Campus_life_coach/finetune/scripts/finetune_all_three_models.py
activity_map = {0 : 'walk', 1 : 'run', 2 : 'sit', 3 : 'stand', 4 : 'lie'}


In [2]:
import torch
import sys
#sys.path.append("Campus_Life_Coach/finetune/src/finetune")
#sys.path.append("/content/Campus_Life_Coach/finetune/src")

#from backbones import build_backbone
#sys.path.append('/content/drive/MyDrive/Colab Notebooks/sliding_window/inference/Campus_Life_Coach-main/finetune/src')
sys.path.append('finetune/src/finetune')
#sys.modules.pop('finetune.models',None)
#sys.path.append("Campus_Life_Coach") # For pretrain there are pre train calls in finetune
from models import SingleStreamClassifier, FusionClassifier, load_pretrained_backbone
# Import in base models
phone_backbone = load_pretrained_backbone("C:/Users/gwenn/Downloads/Campus_Life_Coach-main/pretrain/artifacts/phone_encoder/best.ckpt",device='cpu')
watch_backbone = load_pretrained_backbone("C:/Users/gwenn/Downloads/Campus_Life_Coach-main/pretrain/artifacts/watch_encoder/best.ckpt", device='cpu')
#dummy_backbone = torch.nn.Identity()
phone_model = SingleStreamClassifier(pretrained_backbone=phone_backbone, num_classes =5)
watch_model = SingleStreamClassifier(pretrained_backbone=watch_backbone, num_classes =5)
fuision_model = FusionClassifier(pretrained_phone_backbone=phone_backbone, pretrained_watch_backbone=watch_backbone,num_classes =5)

# Import in weights to model
just_path_f = torch.load("finetune/models/dashboard_models/fusion_classifier.pth",map_location="cpu")
state = just_path_f.get("model_state_dict",just_path_f)
fuision_model.load_state_dict(state, strict=False)
#phone
just_path_p = torch.load("finetune/models/dashboard_models/phone_only_classifier.pth",map_location="cpu")
state_p = just_path_p.get("model_state_dict", just_path_p)
phone_model.load_state_dict(state_p, strict=False)
#watch
just_path_w = torch.load("finetune/models/dashboard_models/watch_only_classifier.pth",map_location="cpu")
state_w = just_path_w.get("model_state_dict",just_path_w)
watch_model.load_state_dict(state_w, strict=False)

# Putting it into inference mode we're telling it to stop updating everything weights etc
fuision_model.eval()
phone_model.eval()
watch_model.eval()


Loading pretrained backbone from C:/Users/gwenn/Downloads/Campus_Life_Coach-main/pretrain/artifacts/phone_encoder/best.ckpt
Checkpoint keys: ['model_state', 'optimizer_state', 'epoch', 'val_metrics', 'config', 'label_names']
Config keys: ['seed', 'device', 'logging', 'optimizer', 'scheduler', 'training', 'augmentation', 'model', 'heads', 'dataset', 'metrics']
  Backbone from config: convnet_small
  Input channels: 6
  Embedding dim: 256
  ✓ Loaded state dict
  ✓ Loaded backbone successfully
Loading pretrained backbone from C:/Users/gwenn/Downloads/Campus_Life_Coach-main/pretrain/artifacts/watch_encoder/best.ckpt
Checkpoint keys: ['model_state', 'optimizer_state', 'epoch', 'val_metrics', 'config', 'label_names']
Config keys: ['seed', 'device', 'logging', 'optimizer', 'scheduler', 'training', 'augmentation', 'model', 'heads', 'dataset', 'metrics']
  Backbone from config: convnet_small
  Input channels: 6
  Embedding dim: 256
  ✓ Loaded state dict
  ✓ Loaded backbone successfully


C:\Users\gwenn\AppData\Local\Temp\ipykernel_6848\3522355824.py:21: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  just_path_f = torch.load("finetune/models/dashboard_models/f

SingleStreamClassifier(
  (encoder): ConvNetBackbone(
    (encoder): Sequential(
      (0): ConvBlock(
        (conv): Conv1d(6, 64, kernel_size=(5,), stride=(1,), padding=(2,), bias=False)
        (bn): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (act): ReLU(inplace=True)
        (dropout): Dropout(p=0.1, inplace=False)
        (shortcut): Sequential(
          (0): Conv1d(6, 64, kernel_size=(1,), stride=(1,), bias=False)
          (1): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        )
      )
      (1): ConvBlock(
        (conv): Conv1d(64, 128, kernel_size=(5,), stride=(2,), padding=(2,), bias=False)
        (bn): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (act): ReLU(inplace=True)
        (dropout): Dropout(p=0.1, inplace=False)
        (shortcut): Sequential(
          (0): Conv1d(64, 128, kernel_size=(1,), stride=(2,), bias=False)
          (1): BatchN

In [3]:
def inference(phone_model,fusion_model,watch_model,phone_window, watch_window):
  # Keep unsqueeze to maintain shape
 # print("In inference now!")
  #print(f"Shape: {phone_window}")
  phone_tensor = torch.tensor(phone_window, dtype=torch.float32).unsqueeze(0).permute(0,2,1)
  watch_tensor = torch.tensor(watch_window, dtype=torch.float32).unsqueeze(0).permute(0,2,1)
# Pass tensor into model and take softmax of logits (log it = model(tensor))
  phone_probs = torch.softmax(phone_model(phone_tensor),dim=1)
  watch_probs = torch.softmax(watch_model(watch_tensor),dim=1)

# Combine them so we get the both the values
  #fusion_input = torch.cat([phone_probs, watch_probs],dim=1)
# Now we are finding the probability of both of them
  fusion_prob = torch.softmax(fuision_model(phone_tensor,watch_tensor),dim=1)
  print(f"reached fuision prob {fusion_prob}")
  return fusion_prob



In [4]:
# Splitting samples and doing inference on each group

def update_samples(phone_samples, watch_samples, phone_sample, watch_sample,phone_model,watch_model,fuision_model):
  for i in range(len(phone_sample)):
    phone_samples.append(phone_sample[i])
  for i in range(len(watch_sample)):
    watch_samples.append(watch_sample[i])
  #phone_samples.append(phone_sample)
  #watch_samples.append(watch_sample)
 # print(f"Phone samples {len(phone_samples)}, Watch samples {len(watch_samples)}, Window Size {WINDOW_SAMPLES}")
  if (len(phone_samples) >= WINDOW_SAMPLES and len(watch_samples) >= WINDOW_SAMPLES):
    phone_window = phone_samples
    watch_window = watch_samples
# Perform inference on group
#Normilazation is done inside classifier
    probabilities = inference(phone_model,fuision_model,watch_model,phone_window, watch_window)
    phone_samples = phone_samples[HOP_SAMPLES:]
    watch_samples = watch_samples[HOP_SAMPLES:]
    print("Should have returned a probability")
    return probabilities, phone_samples, watch_samples
  return None, phone_samples,watch_samples




In [5]:
activity_map = {0 : 'walk', 1 : 'run', 2 : 'sit', 3 : 'stand', 4 : 'lie'}

def most_frequent(list,activity_map):
  walk_count = 0
  run_count = 0
  sit_count = 0
  stand_count = 0
  lie_count = 0
  for i in range(len(list)):
    ele = list[i]
    if ele == 'walk':
      walk_count += 1
    elif ele == 'sit':
      sit_count +=1
    elif ele == 'stand':
      stand_count += 1
    elif ele == 'lie':
      lie_count +=1
    elif ele == 'run':
      run_count +=1
  count_list = [walk_count,run_count,sit_count,stand_count,lie_count]
  return activity_map[count_list.argmax().item()]


In [ ]:
import dash
from dash.dependencies import Output, Input
from dash import dcc, html, dcc
from datetime import datetime
import json
import plotly.graph_objs as go
from collections import deque
from flask import Flask, request
import socket
import time
# ----- Set up Flask + Dash ------
hostname = socket.gethostname()
print(socket.gethostbyname(hostname))

server = Flask(__name__)
app = dash.Dash(__name__, server=server)
# ------------For Dash Graphing-----------------
MAX_DATA_POINTS = 1000
UPDATE_FREQ_MS = 100
time_accel = deque(maxlen=MAX_DATA_POINTS)
time_gyro = deque(maxlen=MAX_DATA_POINTS)

time_d = deque(maxlen=MAX_DATA_POINTS)
avg_accel = deque(maxlen=MAX_DATA_POINTS)
avg_gyro = deque(maxlen=MAX_DATA_POINTS)
x_data = deque(maxlen=MAX_DATA_POINTS)
y_data = deque(maxlen=MAX_DATA_POINTS)
z_data = deque(maxlen=MAX_DATA_POINTS)
# -------------- Layout -----------------------
app.layout = html.Div(
  [
    dcc.Markdown(
      children="""
      # Live Sensor Readings
      Streamed from Sensor Logger: tszheichoi.com/sensorlogger
    """
    ),
    dcc.Graph(id="live_graph"),
    dcc.Interval(id="counter", interval=UPDATE_FREQ_MS),
  ]
)


@app.callback(Output("live_graph", "figure"), Input("counter", "n_intervals"))
def update_graph(_counter):
  accel = list(avg_accel)
  gyro = list(avg_gyro)
  #print(f"Something is up with the graph {accel}, {gyro}")
  t_accel = list(time_accel)
  t_gyro = list(time_gyro)

  data = [
        go.Scatter(x=t_accel, y=accel, name="accelerometer"),
        go.Scatter(x=t_gyro, y=gyro, name="gyroscope"),
    ]

  graph = {
        "data": data,
        "layout": go.Layout(
            xaxis={"type": "date"},
            yaxis={"title": "Sensor Value"},
        ),
    }

    # filter None values for y-range calc
  vals = [v for v in accel + gyro if v is not None]
  if vals:
    graph["layout"]["yaxis"]["range"] = [min(vals), max(vals)]
  all_t = t_gyro + t_accel
  if all_t:
    graph["layout"]["xaxis"]["range"] = [min(all_t), max(all_t)]

  return graph
#------------ IMU for Inference ------------------
from collections import deque
import threading
IMU_BUFFER = deque(maxlen=5000)
buffer_lock = threading.Lock()
#---------------Receive IMU Data ---------------
@server.route("/data", methods=["POST"])
def data():  # listens to the data streamed from the sensor logger
  if str(request.method) == "POST":
    ax = None
    ay = None
    az = None
    gx = None
    gy = None
    gz = None
    #print(f'received data: {request.data}')
    last_acel_time = 0    
    data = json.loads(request.data)
    
    for d in data['payload']:
      if (
        d.get("name", None) == "accelerometer"
      ):  #  modify to access different sensors
        #print("accelerometer received")
        ts = datetime.fromtimestamp(d["time"] / 1000000000)
       # print(f" before any filtering {d}")
        if len(time_accel) == 0 or d["time"] > last_acel_time:
          time_accel.append(ts)
          # modify the following based on which sensor is accessed, log the raw json for guidance
          avg = (d["values"]["x"] + d["values"]["y"] + d["values"]["z"]) / 3
          avg_accel.append(avg)
          #avg_gyro.append(None)
          ax = d["values"]["x"]
          ay = d["values"]["y"]
          az = d["values"]["z"]
          last_acel_time = d["time"]
          #print(f"Current ax IMPORTANT {ax}")
          #current_accel = (ax, ay, az)
      elif (
        d.get("name", None) == "gyroscope"
      ):
        #print("gyroscope received")
        ts = datetime.fromtimestamp(d["time"] / 1000000000)
        if len(time_gyro) == 0 or ts > time_gyro[-1]:
          time_gyro.append(ts)
          # modify the following based on which sensor is accessed, log the raw json for guidance
          avg = (d["values"]["x"] + d["values"]["y"] + d["values"]["z"]) / 3
          avg_gyro.append(avg)
          #avg_accel.append(None)
          gx = d["values"]["x"]
          gy = d["values"]["y"]
          gz = d["values"]["z"]
          #print(f"Current gx IMPORTANT {gx}")
          #current_gyro = (gx, gy, gz)
      #print(f"prints acell and gyro: {ax}, {gx}")
      if None not in [ax,ay,az,gx,gy,gz]:
        with buffer_lock:
          #print("appended")
          IMU_BUFFER.append([ax,ay,az,gx,gy,gz])
  return "success"
#-------- Connects inference and reading in---------
def get_next_sample_block():
    samples = []
    #print(len(IMU_BUFFER))
    with buffer_lock:
        #check if we can start giving samples
        if len(IMU_BUFFER) >= 50:
            #print(f"Sample: {IMU_BUFFER[0]}")
            samples.append(IMU_BUFFER[0])
            for i in range(50):
              IMU_BUFFER.popleft()
    #print("RETURNED A SAMPLE")
    #print(samples)
    return samples

# Set up the PHYPHOX data
print("STARTING MAIN INFERENCE ")
# Set up the PHYPHOX data


#print("Getting data from live IMU >:)")
def inference_loop(phone_model,watch_model,fuision_model):
  #print("STARTING MAIN INFERENCE in function") 
  last_time = None
# For buffer and inference steps
  program_run = True
  phone_samples = []
  watch_samples = []
  predict_probs = []
  counter = 0
# This is for Hysteresis
  runner_up_label = None
  current_label = None
# counter in case we pick a very bad runner up label
  stale_counter = 0
  while(program_run):
    
    counter+= 1
    if (counter == 50):
      user_answered = False
      while(user_answered):
        output = input("Do you want to keep using this? Y for yes N for no")
        if output == 'Y':
          program_run = False
          user_answered = True
        elif output == 'N':
          program_run = True
          user_answered = True
        else:
          print("Please input Y for yes or N for no")
          continue
      #program_run = False
    #print("in loop")
  # Probability Smoothing we are going to use last 3 windows to start
  # we can do probability smooth(rolling average aka average last n probs then take argmax) or Majority vote keep most common label
  # I tested only with phone so we might wanna test with apple watch to test which smoothing technique is the best
  
  #Rolling Average vs Majority wins
    #print(f"Length of predicted probabilities: {len(predict_probs)}")
    if (len(predict_probs) == 3):
     # print("Testing Rolling Average...")
    # Rolling Average ===============================================
      smoothed_probs = sum(predict_probs)/3
      #print(f"SMoothed probs{smoothed_probs}")
      predicted_idx_final = smoothed_probs.argmax().item()
      official_predicted_label = activity_map[predicted_idx_final]
    # Majority wins ===================================================
    #label_vec = []
    #for i in range(len(predict_probs)):
    #  specific_prob = predict_probs[i]
    #  specific_pred_idx = specific_prob.argmax().item()
    #  label_vec.append(activity_map[specific_pred_idx])
    #official_predicted_label = most_frequent(label_vec,activity_map)
    #==================================================================

    # Performing Hysteresis
      #print("PERFORMING HYSTERESIS")
      if(current_label == None):
        #print("YOU SHOULD BE ASSIGNING")
        current_label = official_predicted_label
      else:
      # if it has a current label it has to prove it isn't a mistake
      # Case 0: Predicting normally continue on
        if (current_label == official_predicted_label):
          continue
      # Official label does not equal current label

      # Case 1: we have perdicted this label and the next prediction also predicted this label we are sure
        elif (runner_up_label == official_predicted_label):
          current_label = runner_up_label
          runner_up_label = None
        # Case 2: have not predicted this label in a row yet so we are unsure
        elif (runner_up_label == None):
          runner_up_label = official_predicted_label
        # Case 3: current label is defined we also habe runner up but new official prediction is different from both
        elif(runner_up_label != official_predicted_label):
          stale_counter +=1
          continue
    # If we see runner_up_label not fitting with predictions enough we will reset it
      if (stale_counter >= 2):
        runner_up_label = None

      print(f"WE THINK IT IS {current_label}")



    # get rid of oldest peice of data
      predict_probs.pop(0)
    print(f"so far we think it's: {current_label}")
    while(len(predict_probs) < 3):
      #print("IN inner loops")
      new_samples_p = get_next_sample_block()
      #----------------FOR WATCH ------------------
      # new_samples_w = get_next_sample_block_w()
      # This is just for testing with phone alone  switch to above same logic code is in watch.py
      new_samples_w = new_samples_p
    
    # Ease of testing
      #print(f"Give me what i need {new_samples_p} and {new_samples_w}")
      if not new_samples_p or not new_samples_w:
        #print("In here never runs?")
        time.sleep(0.01)
        continue
      phone_sample = new_samples_p
      watch_sample = new_samples_w
      #phone
      #print(f"Phone sample: {phone_sample} or {new_samples_p}")
      #print("About to get probability distri.")
      prob,p_sample,w_sample = update_samples(phone_samples,watch_samples,phone_sample,watch_sample,phone_model,watch_model,fuision_model)
      #print("Gotten probability distri.")
      phone_samples = p_sample
      watch_samples = w_sample
    # If not none we know we can make a prediction!
      if prob != None:
  # We should have a probability distribution now of both modalities combined!
        predicted_idx = prob.argmax().item()
        predicted_label = activity_map[predicted_idx]
        print(f"You MIGHT be doing this {predicted_label}")
        predict_probs.append(prob)
      else:
        continue






#---- Start --------------------

if __name__ == "__main__":
    # This is so inference loop can run look up threading documentation for more info
    t = threading.Thread(target=inference_loop,args=(phone_model,watch_model,fuision_model),daemon=True)
    t.start()
    
    app.run(port=8000, host="10.105.167.23")

10.105.167.23
STARTING MAIN INFERENCE 
so far we think it's: None


reached fuision prob tensor([[0.6868, 0.0060, 0.0026, 0.2979, 0.0067]], grad_fn=<SoftmaxBackward0>)
Should have returned a probability
You MIGHT be doing this walk
reached fuision prob tensor([[0.4917, 0.4707, 0.0022, 0.0338, 0.0015]], grad_fn=<SoftmaxBackward0>)
Should have returned a probability
You MIGHT be doing this walk
reached fuision prob tensor([[7.0656e-01, 8.5928e-02, 2.3617e-04, 2.0601e-01, 1.2655e-03]],
       grad_fn=<SoftmaxBackward0>)
Should have returned a probability
You MIGHT be doing this walk
WE THINK IT IS walk
so far we think it's: walk
reached fuision prob tensor([[5.8335e-01, 1.0920e-01, 4.5928e-04, 3.0506e-01, 1.9278e-03]],
       grad_fn=<SoftmaxBackward0>)
Should have returned a probability
You MIGHT be doing this walk


In [ ]:
# Set up the PHYPHOX data

#PHYPHOX_URL = "http://172.20.10.1:8080/experiment"
#print("Getting data from live IMU >:)")
def inference_loop(): 
  last_time = None
# For buffer and inference steps
  program_run = True
  phone_samples = []
  watch_samples = []
  predict_probs = []

# This is for Hysteresis
  runner_up_label = None
  current_label = None
# counter in case we pick a very bad runner up label
  stale_counter = 0
  while(program_run):
  # Probability Smoothing we are going to use last 3 windows to start
  # we can do probability smooth(rolling average aka average last n probs then take argmax) or Majority vote keep most common label
  # I tested only with phone so we might wanna test with apple watch to test which smoothing technique is the best
  
  #Rolling Average vs Majority wins
    if (len(predict_probs) == 3):
    # Rolling Average ===============================================
      smoothed_probs = sum(predict_probs)/3
      predicted_idx_final = smoothed_probs.argmax().item()
      official_predicted_label = activity_map[predicted_idx_final]
    # Majority wins===================================================
    #label_vec = []
    #for i in range(len(predict_probs)):
    #  specific_prob = predict_probs[i]
    #  specific_pred_idx = specific_prob.argmax().item()
    #  label_vec.append(activity_map[specific_pred_idx])
    #official_predicted_label = most_frequent(label_vec,activity_map)
    #==================================================================

    # Performing Hysteresis
      if(current_label == None):
        curent_label = official_predicted_label
      else:
      # if it has a current label it has to prove it isn't a mistake
      # Case 0: Predicting normally continue on
        if (current_label == official_predicted_label):
          continue
      # Official label does not equal current label

      # Case 1: we have perdicted this label and the next prediction also predicted this label we are sure
        elif (runner_up_label == official_predicted_label):
          current_label = runner_up_label
          runner_up_label = None
        # Case 2: have not predicted this label in a row yet so we are unsure
        elif (runner_up_label == None):
          runner_up_label = official_predicted_label
        # Case 3: current label is defined we also habe runner up but new official prediction is different from both
        elif(runner_up_label != official_predicted_label):
          stale_counter +=1
          continue
    # If we see runner_up_label not fitting with predictions enough we will reset it
      if (stale_counter >= 2):
        runner_up_label = None

      print(f"Actual label predicted {current_label}")



    # get rid of oldest peice of data
      predict_probs.pop(0)

    while(len(predict_probs)< 3):
      new_samples = get_next_sample_block()
    #phone_sample, last_time_return = get_phyphonx_data(PHYPHOX_URL, last_time)
    # Ease of testing
      if not new_samples:
        time.sleep(0.01)
        continue
      for i in range(len(new_samples)):
        phone_sample = new_samples[i]
        watch_sample = new_samples[i]

      prob,p_sample,w_sample = update_samples(phone_samples,watch_samples,phone_sample,watch_sample)
    # If not none we know we can make a prediction!
      if prob != None:
  # We should have a probability distribution now of both modalities combined!
        predicted_idx = prob.argmax().item()
        predicted_label = activity_map[predicted_idx]
        print(f"not actual label just tests {predicted_label}")
        predict_probs.append(prob)
      else:
        continue


In [ ]:
# Live reading in data